<a id="top"></a>

# Demo: temperature is a knob, not a switch

```yaml
title: "Demo: temperature is a knob, not a switch"
keywords: temperature, sampling, top-p, top-k, variance, determinism, teacher demo
estimated duration: 15
```

> **Lesson:** L01. Teacher demo — Demo 3 in the [roadmap](../../../../docs/origin/lesson_roadmaps/L01/demos_or_activities.md). Uses a canned client so it runs offline and deterministically; an optional gated cell makes real calls if `ANTHROPIC_API_KEY` is set.

## Contents

- [1. Setup](#1-setup)
- [2. Low vs. high temperature](#2-low-vs-high-temperature)
- [3. Try it live (optional)](#3-try-it-live-optional)
- [4. Other knobs, and low-not-zero variance](#4-other-knobs-and-low-not-zero-variance)

## 1. Setup

Show the distribution diagram (slide) *before* running anything.

In [1]:
import os

from fluffy_potato_curriculum.potato_llm import AnthropicClient, ChatResponse, Message, Usage


class CannedLLM:
    """An offline stand-in for a real client, so this notebook is deterministic.

    It fakes the *behavior* we want to observe: at low temperature it commits to one
    answer; at higher temperature it varies. Because it matches the `PotatoLLMClient`
    shape (a `model` and a `chat` method), the optional live cell can swap in a real
    `AnthropicClient` with no other changes.
    """

    model = "canned-sonnet"

    def __init__(self) -> None:
        self._varied = [
            "Rainy Day Roasters",
            "The Drizzle",
            "Petrichor Coffee",
            "Cloudbreak Beans",
            "Umbrella & Bean",
        ]
        self._calls = 0

    def chat(
        self,
        messages: list[Message],
        *,
        max_tokens: int = 1024,
        temperature: float = 1.0,
    ) -> ChatResponse:
        # Low temperature => sharp distribution => always the top choice.
        # Higher temperature => flatter => cycle through alternatives (deterministically).
        text = (
            self._varied[0] if temperature < 0.5 else self._varied[self._calls % len(self._varied)]
        )
        self._calls += 1
        return ChatResponse(
            text=text, model=self.model, usage=Usage(input_tokens=20, output_tokens=4), raw=None
        )


PROMPT = "Suggest a name for a coffee shop on a rainy street in Seattle. Just the name, one or two words."
print("setup ready (offline, canned client)")

setup ready (offline, canned client)


[↑ Back to top](#top)

## 2. Low vs. high temperature

Same prompt, two temperatures. Low commits to one answer; high spreads out.

In [2]:
def sample_n(client: CannedLLM, temperature: float, n: int) -> list[str]:
    """Run the prompt n times at a temperature; return the replies."""
    return [
        client.chat([Message.user(PROMPT)], max_tokens=16, temperature=temperature).text
        for _ in range(n)
    ]


client = CannedLLM()
print("temperature=0 (near-identical):")
for line in sample_n(client, 0.0, 5):
    print("  ", line)
print("\ntemperature=1 (varied):")
for line in sample_n(client, 1.0, 5):
    print("  ", line)

temperature=0 (near-identical):
   Rainy Day Roasters
   Rainy Day Roasters
   Rainy Day Roasters
   Rainy Day Roasters
   Rainy Day Roasters

temperature=1 (varied):
   Rainy Day Roasters
   The Drizzle
   Petrichor Coffee
   Cloudbreak Beans
   Umbrella & Bean


[↑ Back to top](#top)

## 3. Try it live (optional)

Swap the canned client for a real one. Gated on `ANTHROPIC_API_KEY`; clear this cell's output before committing.

In [3]:
if os.environ.get("ANTHROPIC_API_KEY"):
    live = AnthropicClient()
    print("live temperature=0:")
    for _ in range(5):
        print("  ", live.chat([Message.user(PROMPT)], max_tokens=16, temperature=0.0).text.strip())
else:
    print("Set ANTHROPIC_API_KEY to run this live. (Skipped — see the canned output above.)")

Set ANTHROPIC_API_KEY to run this live. (Skipped — see the canned output above.)


[↑ Back to top](#top)

## 4. Other knobs, and low-not-zero variance

- Temperature reshapes the distribution *before* sampling; low = the top token dominates.
- `top_p` (nucleus) and `top_k` restrict the *candidate set* instead — different lever. We name
  them but don't tune them in this course.
- Temperature 0 is **low** variance, not **zero**: floating-point math, batching, and tie-breaks
  leak through. Set that expectation now so a future flaky eval doesn't read as a bug.

[↑ Back to top](#top)